[//]: # (cr:doc name='chapter_3_silver_merge' id=4f395246)
# Chapter 3: Silver Merge

**Purpose:** Merge all Bronze-stage datasets onto a temporal spine (`entity_id` x `as_of_date`), producing a unified Silver feature matrix aligned on the SnapshotGrid.

**When to use this notebook:**
- Notebooks 00–01d have run and produced Bronze datasets in the RunNamespace
- A SnapshotGrid has been defined (notebook 01 or 01c)
- You want to create a single wide table aligned on `(entity_id, as_of_date)` for downstream analysis

**What you'll learn:**
- How to build a temporal spine from the SnapshotGrid
- How event datasets equi-join on `(entity_id, as_of_date)`
- How entity datasets broadcast across all observation dates
- How to validate temporal integrity of the merged result

**Outputs:**
- Silver merged Delta table at `namespace.silver_merged_path`
- Merge report (row counts, column counts, conflict resolutions)
- Temporal integrity validation

---

## Spine Table Pattern

```
SnapshotGrid.grid_dates    All entity_ids from Bronze datasets
        |                              |
        v                              v
   [2024-01-01,          x       [A, B, C, ...]
    2024-02-01,                         |
    2024-03-01, ...]                    |
        |                              |
        +--------- CROSS PRODUCT ------+
                       |
                       v
              SPINE (entity_id, as_of_date)
                       |
        +--------------+--------------+
        |              |              |
   LEFT JOIN      LEFT JOIN      LEFT JOIN
   (equi-join)    (broadcast)    (as-of)
        |              |              |
   Event Bronze   Entity Bronze  Entity w/ timestamp
   (has as_of)   (no as_of)     (feature_timestamp)
        |              |              |
        +--------> SILVER MERGED <----+
```

[//]: # (cr:doc name='3_1_setup' id=868134b0)
## 3.1 Setup

In [ ]:
# @cr:code name='init_progress' id=9e2ad1d4
from customer_retention.analysis.notebook_progress import accept_workflow_params, track_and_export_previous
from customer_retention.analysis.visualization import display_table

accept_workflow_params()
track_and_export_previous("03_dataset_merge.ipynb")

from customer_retention.analysis.auto_explorer import mark_notebook
from customer_retention.analysis.auto_explorer.active_dataset_store import (
    load_active_dataset,
    load_merge_dataset,
    load_merge_dataset_distributed,
)
from customer_retention.analysis.auto_explorer.key_resolver import resolve_entity_keys
from customer_retention.analysis.auto_explorer.project_context import ProjectContext, RawTimeColumnRole
from customer_retention.analysis.auto_explorer.run_namespace import RunNamespace
from customer_retention.analysis.auto_explorer.snapshot_grid import SnapshotGrid
from customer_retention.core.compat import (
    concat,
    native_pd,
    pd,
    safe_describe,
    safe_memory_usage_bytes,
    track_stage_object,
    use_distributed_processing,
)
from customer_retention.core.config.column_config import DatasetGranularity
from customer_retention.core.config.experiments import (
    FINDINGS_DIR,
)
from customer_retention.integrations.adapters.factory import get_delta
from customer_retention.stages.temporal import (
    DatasetMergeInput,
    MergeConfig,
    SparkTemporalMerger,
    TemporalMerger,
)

# --- cr:profiler ---
if __import__('os').environ.get("CR_BATCH_EXECUTION") == "1":
    import json as _j
    import os as _os
    import re as _r
    _cr_nb = _os.path.splitext(_os.path.basename(_os.environ.get("PAPERMILL_OUTPUT_PATH", "")))[0]
    if _cr_nb:
        _cr_mp = _os.path.join(_os.getcwd(), f".cr_cell_metrics_{_cr_nb}.jsonl")
        open(_cr_mp, 'w').close()
        _cr_re = _r.compile(r"^#\s*@cr:\w+\s+name='([^']+)'\s+id=(\w+)")
        def _cr_jc():
            return -1
        try:
            _s = __import__('pyspark.sql', fromlist=['SparkSession']).SparkSession.getActiveSession()
            if _s:
                def _cr_jc():  # noqa: F811
                    return _s._jsc.sc().dagScheduler().nextJobId().get()
        except Exception:
            pass
        def _cr_pre(info):
            info._cr_sj = _cr_jc()
        def _cr_post(r):
            sj = getattr(r.info, '_cr_sj', -1)
            sa = _cr_jc()
            m = _cr_re.match((r.info.raw_cell or '').split('\n')[0])
            if m:
                with open(_cr_mp, 'a') as f:
                    f.write(_j.dumps({"cell_name": m.group(1), "cell_id": m.group(2),
                                      "spark_jobs": (sa - sj) if sj >= 0 and sa >= 0 else None}) + '\n')
        get_ipython().events.register('pre_run_cell', _cr_pre)
        get_ipython().events.register('post_run_cell', _cr_post)
# --- /cr:profiler ---


[//]: # (cr:doc name='3_2_load_context' id=845d75cf)
## 3.2 Load Context

In [ ]:
# @cr:code name='load_namespace' id=b45f7959
_namespace = RunNamespace.from_env_or_latest()
if _namespace is None:
    raise RuntimeError("No RunNamespace found. Run notebook 00 first.")
mark_notebook(_namespace, "03_dataset_merge.ipynb")

context = ProjectContext.load(_namespace.project_context_path)
grid = SnapshotGrid.load(_namespace.snapshot_grid_path)

print(f"Run: {_namespace.run_id}")
print(f"Project: {context.project_name}")
print(f"Datasets registered: {len(context.datasets)}")
print(f"Grid dates: {len(grid.grid_dates)} ({grid.cadence_interval.value} cadence)")
if grid.grid_dates:
    print(f"  Range: {grid.grid_dates[0]} to {grid.grid_dates[-1]}")

[//]: # (cr:doc name='3_3_load_bronze_datasets' id=43392bf0)
## 3.3 Load Bronze Datasets

In [ ]:
# @cr:code name='prepare_merge_inputs' id=629fca83
dataset_names = sorted(context.datasets.keys())
merge_inputs: list[DatasetMergeInput] = []
entity_id_parts: list = []

_use_distributed = use_distributed_processing()

print("=" * 70)
print("BRONZE DATASETS")
print("=" * 70 + "\n")

_load = load_merge_dataset_distributed if _use_distributed else load_merge_dataset
loaded_frames: dict = {}
for name in dataset_names:
    entry = context.datasets.get(name)
    granularity = entry.granularity if entry else DatasetGranularity.UNKNOWN
    loaded_frames[name] = _load(_namespace, name, granularity)

resolutions = {
    name: context.datasets[name].key_resolution
    for name in dataset_names
    if context.datasets.get(name) and context.datasets[name].key_resolution
}
if resolutions:
    loaded_frames = resolve_entity_keys(loaded_frames, resolutions)
    for ds_name, steps in resolutions.items():
        print(f"  Key resolution: {ds_name} via {' -> '.join(s.bridge_dataset for s in steps)}")

for name in dataset_names:
    df = loaded_frames[name]
    entry = context.datasets.get(name)
    granularity = entry.granularity if entry else DatasetGranularity.UNKNOWN
    entity_col = entry.entity_column if entry else None
    feature_ts = (
        entry.time_column
        if (entry and granularity == DatasetGranularity.ENTITY_LEVEL
            and entry.time_column
            and entry.raw_time_column_role != RawTimeColumnRole.ENTITY_UPDATE_TIME)
        else None
    )

    if entity_col and entity_col in df.columns:
        entity_id_parts.append(df[entity_col])
        if entity_col != "entity_id":
            df = df.rename(columns={entity_col: "entity_id"})

    merge_inputs.append(DatasetMergeInput(
        name=name,
        df=df,
        granularity=granularity,
        feature_timestamp_column=feature_ts,
    ))

    label = granularity.value if granularity else "unknown"
    print(f"  {name}")
    print(f"    Granularity: {label}")
    print(f"    Shape: {df.shape[0]:,} rows x {df.shape[1]} cols")
    if entity_col and entity_col != "entity_id":
        print(f"    Entity column: {entity_col} -> entity_id")
    if feature_ts:
        print(f"    Feature timestamp: {feature_ts}")
    print()

if dataset_names and not entity_id_parts:
    missing = [n for n in dataset_names
               if not (context.datasets.get(n) and context.datasets[n].entity_column
                       and context.datasets[n].entity_column in loaded_frames[n].columns)]
    raise ValueError(
        f"No entity columns found in loaded datasets. "
        f"Check entity_column in ProjectContext for: {missing}"
    )

del loaded_frames
all_entity_ids = concat(entity_id_parts, ignore_index=True) if entity_id_parts else native_pd.Series(dtype=str)
print(f"Total datasets loaded: {len(merge_inputs)}")

[//]: # (cr:doc name='3_4_build_spine' id=b527c771)
## 3.4 Build Spine

In [ ]:
# @cr:code name='configure_merger' id=25c42a4c
config = MergeConfig(entity_key="entity_id")
merger = SparkTemporalMerger(config=config) if _use_distributed else TemporalMerger(config=config)

spine = merger.build_spine(all_entity_ids, grid.grid_dates)

print("=" * 70)
print("SPINE")
print("=" * 70)
print(f"  Unique entities: {spine['entity_id'].nunique():,}")
print(f"  Grid dates:      {spine['as_of_date'].nunique()}")
print(f"  Total rows:      {len(spine):,}")
_spine_bytes = safe_memory_usage_bytes(spine)
if _spine_bytes:
    print(f"  Estimated size:  {_spine_bytes / 1024 / 1024:.1f} MB (spine only)")

[//]: # (cr:doc name='3_5_merge' id=9ae579cb)
## 3.5 Merge

In [ ]:
# @cr:code name='execute_merge' id=8f591bea
if _use_distributed:
    from customer_retention.analysis.auto_explorer.active_dataset_store import merge_datasets_incremental
    from customer_retention.core.compat import as_spark_df

    spine_sdf = as_spark_df(spine)
    report = merge_datasets_incremental(
        namespace=_namespace,
        spine_sdf=spine_sdf,
        datasets=merge_inputs,
        merger=merger,
    )
    del spine_sdf
    merged = None  # will be read from Delta in the save cell
else:
    merged, report = merger.merge_all(spine, merge_inputs)

print("=" * 70)
print("MERGE RESULTS")
print("=" * 70)
print(f"  Datasets merged:  {len(report.datasets_merged)}")
if merged is not None:
    print(f"  Final shape:      {merged.shape[0]:,} rows x {merged.shape[1]} cols")
else:
    print(f"  Final shape:      {report.spine_rows:,} rows x {report.total_columns} cols")
print()

print("Per-dataset column counts:")
for ds_name, n_cols in report.columns_per_dataset.items():
    print(f"  {ds_name}: +{n_cols} columns")

if report.renamed_columns:
    print(f"\nColumn conflicts resolved: {len(report.renamed_columns)}")
    for original, renamed in list(report.renamed_columns.items())[:10]:
        print(f"  {original}")

[//]: # (cr:doc name='3_6_validation' id=f41f80f6)
## 3.6 Validation

In [ ]:
# @cr:code name='check_temporal_integrity' id=21c42493
print("=" * 70)
print("TEMPORAL INTEGRITY")
print("=" * 70)

ti = report.temporal_integrity
if ti:
    status = "PASS" if ti.get("valid", True) else "FAIL"
    print(f"  Status: {status}")
    issues = ti.get("issues", [])
    if issues:
        for issue in issues:
            print(f"  Issue: {issue.get('type', 'unknown')} - {issue.get('message', '')}")
    else:
        print("  No temporal integrity issues detected.")
else:
    print("  Temporal validation was skipped.")

if merged is not None:
    _actual_rows = len(merged)
else:
    from customer_retention.core.compat.detection import get_spark_session
    _actual_rows = get_spark_session().read.format("delta").load(
        str(_namespace.silver_merged_path)
    ).count()
assert _actual_rows == report.spine_rows, (
    f"Row count mismatch: merged={_actual_rows}, spine={report.spine_rows}"
)
print(f"\n  Spine row count preserved: {_actual_rows:,}")

[//]: # (cr:doc name='3_7_save_merged_dataset' id=52fb25de)
## 3.7 Save Merged Dataset

In [ ]:
# @cr:code name='save_silver_merged' id=2887f9d7
import gc
import time as _time

output_path = _namespace.silver_merged_path
delta = get_delta() if _use_distributed else get_delta(force_local=True)

print(f"Output path: {output_path}")
print(f"  Adapter: {type(delta).__name__}")

_t0 = _time.monotonic()

del merge_inputs, spine, all_entity_ids, entity_id_parts
gc.collect()
print(f"  [{_time.monotonic() - _t0:.1f}s] Intermediate frames released")

if _use_distributed:
    # merge_datasets_incremental already wrote + optimised the Delta table
    print(f"  [{_time.monotonic() - _t0:.1f}s] Delta write+optimize already done (incremental merge)")
else:
    print(f"  DataFrame: {type(merged).__name__} {merged.shape}")
    _merged_bytes = safe_memory_usage_bytes(merged)
    if _merged_bytes:
        print(f"  Memory: {_merged_bytes / 1024**2:.0f} MB")

    delta.write(merged, str(output_path), mode="overwrite")
    print(f"  [{_time.monotonic() - _t0:.1f}s] Delta write complete")
    del merged
    gc.collect()

    _z_cols = [c for c in ["entity_id", "as_of_date"] if c in delta.read(str(output_path)).columns]
    if _z_cols:
        delta.optimize(str(output_path), _z_cols)
    else:
        delta.optimize(str(output_path))
    print(f"  [{_time.monotonic() - _t0:.1f}s] OPTIMIZE complete")

print(f"Silver merged dataset saved to: {output_path}")
print(f"  Shape: {report.spine_rows:,} rows x {report.total_columns} cols")

from customer_retention.analysis.auto_explorer.findings import ExplorationFindings

all_findings_paths = _namespace.discover_all_findings(prefer_aggregated=True)
dataset_findings = [ExplorationFindings.load(str(p)) for p in all_findings_paths]
merged_findings = ExplorationFindings.merge_from_datasets(
    dataset_findings,
    row_count=report.spine_rows,
    column_count=report.total_columns,
    source_path=str(output_path),
    renamed_columns=report.renamed_columns,
    entity_key=config.entity_key,
)

merged = delta.read(str(output_path))
track_stage_object(merged)
_type_corrections = merged_findings.reconcile_column_types(merged)
if _type_corrections:
    print(f"\n  Type reconciliation: {len(_type_corrections)} column(s) corrected")
    for _col_name in _type_corrections:
        print(f"    {_col_name}: reclassified as {merged_findings.columns[_col_name].inferred_type.value}")

_namespace.merged_dir.mkdir(parents=True, exist_ok=True)
merged_findings.save(str(_namespace.merged_findings_path))

print(f"  Merged findings saved to: {_namespace.merged_findings_path}")
print(f"  Columns catalogued: {len(merged_findings.columns)}")
print(f"  [{_time.monotonic() - _t0:.1f}s] Cell complete")

[//]: # (cr:doc name='3_8_preview' id=927a85f6)
## 3.8 Preview

In [ ]:
# @cr:code name='preview_merged_data' id=890ee6f2
print("First 10 rows:")
display_table(merged.head(10))

print("\nDescriptive statistics:")
display_table(safe_describe(merged))

[//]: # (cr:doc name='3_9_summary' id=c36cadb1)
## 3.9 Summary

In [ ]:
# @cr:code name='display_merge_summary' id=cde7b56b
print("=" * 70)
print("SILVER MERGE SUMMARY")
print("=" * 70)
print(f"  Spine: {report.spine_entities:,} entities x {report.spine_dates} dates = {report.spine_rows:,} rows")
print(f"  Datasets merged: {', '.join(report.datasets_merged)}")
print(f"  Total columns: {report.total_columns}")
if report.renamed_columns:
    print(f"  Renamed columns: {len(report.renamed_columns)}")
print(f"  Output: {output_path}")
print()
print("Next steps:")
print("  04_column_deep_dive.ipynb  - Feature deep dive on merged data")
print("  05_relationship_analysis.ipynb - Correlation and interaction analysis")

In [ ]:
# @cr:code name='release_stage_memory' id=b3c4e925
from customer_retention.core.compat import release_stage_memory

release_stage_memory()
del merged

[//]: # (cr:doc name='summary_what_we_learned' id=6cf6a86d)
---

## Summary: What We Learned

In this notebook, we:

1. **Loaded Context** - RunNamespace, ProjectContext, and SnapshotGrid
2. **Loaded Bronze Datasets** - All datasets from the namespace with their granularity metadata
3. **Built Spine** - Cross product of all entity IDs and grid dates
4. **Merged** - Left-joined all datasets onto the spine (event equi-join, entity broadcast)
5. **Validated** - Checked temporal integrity of the merged result
6. **Saved** - Silver merged Delta table for downstream notebooks

---

## Next Steps

Continue to **04_column_deep_dive.ipynb** to:
- Deep dive into individual columns of the merged dataset
- Analyze distributions, outliers, and missing patterns

Or **05_relationship_analysis.ipynb** to:
- Explore correlations between features on the merged data
- Analyze feature-target relationships
- Detect multicollinearity

[//]: # (cr:doc name='section' id=49724b2f)
> **Save Reminder:** Save this notebook (Ctrl+S / Cmd+S) before running the next one.
> The next notebook will automatically export this notebook's HTML documentation from the saved file.